### RoPE

Rotary Position Embedding 

In [4]:
import torch.nn as nn
import torch
import math

In [5]:
class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0 # the head_dim must be even for the RoPE

        freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))

        # angle for every (position, freq) pairs:
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)

        # cache as buffers so they move with .to(device) and are not trained
        self.register_buffer("cos", torch.cos(angles), persistent = False)
        self.register_buffer("sin", torch.sin(angles), persistent = False)

    def forward(self, T):
        return self.cos[:T], self.sin[:T]

def apply_rope(q, k, rope_cos, rope_sin):
    cos = rope_cos.unsqueeze(0).unsqueeze(0)
    sin = rope_sin.unsqueeze(0).unsqueeze(0) # [1, 1, T, head_dim/2]

    def rotate(x):
        x1 = x[..., 0::2] # even dim
        x2 = x[..., 1::2] # odd dim 

        rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim = -1)

        return rotated.flatten(-2) # flatten back to [B, H, T, head_dim]

    return rotate(q), rotate(k)